# Open Legal Chile — Cuaderno Reproducible de Jurimetría e Investigación Empírica (DuckDB + Apache Parquet + LegalGraphify)

**Open Legal Chile — Un proyecto open source · Desarrollado por Pablo Benavides J. · Web: https://elpabloultron.github.io/open-legal-chile/**

Este cuaderno reproducible permite a investigadores, tesistas y académicos del Derecho consultar en segundos las **5 tablas Apache Parquet** del corpus jurídico chileno (`pablobenavidesj/doctrina-jurisprudencia-chile`) con **DuckDB SQL**, explorar la red multiplex de citaciones (`PageRank` dirigido + `HITS Authority/Hub`), cuantificar el **Baremo Estadístico de Daño Moral PJUD (UF/CLP)** y calcular **métricas de validación humana intercodificador** (*Precisión*, *Cobertura*, *F1* y **coeficiente $\kappa$ de Cohen**).

> **Nota metodológica (Derecho Continental Chileno):** Conforme al Art. 1 y Art. 3 inc. 2 del Código Civil, la jurisprudencia carece de fuerza obligatoria general (*stare decisis* inexistente). Los indicadores cuantitativos de este cuaderno miden convergencia argumentativa empírica y distribución estadística de fallos.

In [ ]:
# 1. Configuración de origen de datos (Parquet local si existe, o Hugging Face remoto vía DuckDB)
from pathlib import Path
import duckdb

REPO_HF = "hf://datasets/pablobenavidesj/doctrina-jurisprudencia-chile/data/parquet"
LOCAL_PARQUET = Path("../data/parquet") if Path("../data/parquet").exists() else Path("data/parquet")

def tabla_uri(nombre: str) -> str:
    local = LOCAL_PARQUET / f"{nombre}.parquet"
    return str(local.resolve()) if local.exists() else f"{REPO_HF}/{nombre}.parquet"

con = duckdb.connect()
print("Origen activo:", "LOCAL -> " + str(LOCAL_PARQUET.resolve()) if (LOCAL_PARQUET / "corpus_entradas.parquet").exists() else REPO_HF)

In [ ]:
# 2. Distribución longitudinal de la Corte Suprema por Sala y resultado normalizado (N = 70.523)
sql_salas = f"""
SELECT
    sala,
    COUNT(*) AS total_fallos,
    SUM(CASE WHEN resultado_normalizado = 'acoge' THEN 1 ELSE 0 END) AS acoge,
    SUM(CASE WHEN resultado_normalizado = 'rechaza' THEN 1 ELSE 0 END) AS rechaza,
    SUM(CASE WHEN resultado_normalizado = 'confirma' THEN 1 ELSE 0 END) AS confirma,
    SUM(CASE WHEN resultado_normalizado = 'revoca' THEN 1 ELSE 0 END) AS revoca,
    SUM(CASE WHEN resultado_normalizado = 'inadmisible' THEN 1 ELSE 0 END) AS inadmisible
FROM '{tabla_uri("corpus_entradas")}'
WHERE col = 'cs' AND sala != ''
GROUP BY sala
ORDER BY total_fallos DESC
LIMIT 10
"""
print(con.execute(sql_salas).fetchall())

In [ ]:
# 3. Baremo Estadístico de Daño Moral por Muerte (PJUD) — Mediana y Rango Intercuartil (UF)
sql_baremo = f"""
SELECT
    parentesco,
    total_casos,
    ROUND(p25_uf_pi, 1) AS p25_uf,
    ROUND(mediana_uf_pi, 1) AS mediana_pi_uf,
    ROUND(p75_uf_pi, 1) AS p75_uf,
    ROUND(mediana_uf_ca, 1) AS mediana_ca_uf,
    ROUND(mediana_uf_cs, 1) AS mediana_cs_uf
FROM '{tabla_uri("baremo_dano_moral")}'
ORDER BY total_casos DESC
"""
for fila in con.execute(sql_baremo).fetchall():
    print(fila)

In [ ]:
# 4. Red Multiplex de Citaciones: Top 15 Normas Positivas Chilenas (excluyendo ruido OCR fuera de rango)
sql_normas = f"""
SELECT
    n.id,
    n.label,
    n.grado_sustantivo,
    ROUND(n.pagerank_sustantivo, 6) AS pr_sustantivo
FROM '{tabla_uri("grafo_nodos")}' n
WHERE n.node_type IN ('norma', 'articulo_legal')
  AND n.rango_norma_plausible = TRUE
ORDER BY n.grado_sustantivo DESC
LIMIT 15
"""
for fila in con.execute(sql_normas).fetchall():
    print(fila)

In [ ]:
# 5. Validación Humana Intercodificador (Cálculo reproducible de Coeficiente kappa de Cohen y F1)
from collections import Counter

def cohen_kappa(cod1: list[str], cod2: list[str]) -> float:
    assert len(cod1) == len(cod2) and len(cod1) > 0
    n = len(cod1)
    po = sum(1 for a, b in zip(cod1, cod2, strict=True) if a == b) / n
    c1, c2 = Counter(cod1), Counter(cod2)
    etiquetas = set(c1) | set(c2)
    pe = sum((c1[e] / n) * (c2[e] / n) for e in etiquetas)
    return round((po - pe) / (1.0 - pe), 4) if pe < 1.0 else 1.0

# Ejemplo con plantilla generada por `jurimetria_analizar_corpus(modo='muestra_validacion', n_muestra=100, semilla=42)`
revisor_a = ["confirma", "revoca", "rechaza", "acoge", "inadmisible", "confirma", "revoca", "rechaza", "confirma", "acoge"]
revisor_b = ["confirma", "revoca", "rechaza", "acoge", "inadmisible", "confirma", "confirma", "rechaza", "confirma", "acoge"]
print("Kappa de Cohen (piloto):", cohen_kappa(revisor_a, revisor_b))